# Fine-tuning Cross-Encoder Reranker on Domain-Specific Data

## Section 1 — Introduction

Cross-encoder fine-tuning involves training a pre-trained sentence transformer model to better predict the relevance of a document chunk given a specific query. Unlike bi-encoders (which encode query and document separately and compute cosine similarity), cross-encoders pass the query and document simultaneously through the transformer network, allowing the attention mechanism to model the interactions between the query and document comprehensively. This results in much higher accuracy but is computationally heavier, which is why it's typically used as a second-stage reranker.

**Why general-purpose rerankers underperform on domain-specific data:**
General-purpose rerankers (like the default `ms-marco-MiniLM-L-6-v2`) are trained on massive open-domain datasets like MS MARCO (Bing search queries). They tend to struggle with domain-specific jargon, proprietary terminology, and highly specialized contexts where relevance isn't solely based on general semantic similarity but on specific technical details or internal knowledge structures.

**What we expect to improve:**
By fine-tuning the reranker on a small set of domain-specific synthetic data, we expect the model to learn our specific vocabularies and relevance criteria. Even with a small dataset, adapting the last few layers of the cross-encoder should lead to an improved Mean Reciprocal Rank (MRR) for our specific queries, pushing the truly relevant documents to the top of the context window.

## Section 2 — Training data generation

In this section, we generate synthetic training data using Gemini 2.0 Flash. We will use a sample document string and ask the LLM to generate 60 training pairs: 30 positive (relevant) queries for specific chunks, and 30 negative (irrelevant) combinations. Each pair will have a query, a document chunk, and a relevance label of 1 for relevant or 0 for irrelevant. We will then save this balanced dataset to `data/reranker_training.json`.

In [1]:
from google import genai
import json
import os
import random

# Ensure data directory exists
os.makedirs("../data", exist_ok=True)

# NOTE: Set your GEMINI_API_KEY environment variable before running this cell.
api_key = os.environ.get("GEMINI_API_KEY")
client = None
if api_key:
    client = genai.Client(api_key=api_key)
else:
    print("Warning: GEMINI_API_KEY not found. Ensure it is set.")

# Sample domain-specific document about an imaginary internal tool 'NexusData'
sample_chunks = [
    "NexusData is an internal data warehousing solution that aggregates telemetry from all edge computing devices. We introduced NexusData in Q3 to replace the legacy Hadoop cluster.",
    "To connect to NexusData, developers must use the `nexus-cli` tool. Authentication requires a short-lived IAM token generated via the `auth.nexus` endpoint.",
    "The primary database engine behind NexusData is optimized for time-series data, meaning analytical queries over last week's traffic will execute in sub-seconds. However, point-in-time updates are discouraged.",
    "Error code NX-503 indicated a rate limiting violation on the ingestion API. If you see NX-503, implement exponential backoff with a maximum retry window of 30 seconds.",
    "For generating daily reports, the `AggregatorTask` should be scheduled using Airflow. It runs at 02:00 UTC and dumps the aggregated CSVs into the cold-storage bucket."
]

prompt = f"""
You are an expert dataset creator for training search and retrieval systems.
Given the following document chunks, generate a JSON array of precisely 60 training pairs.
    "Generate 30 positive pairs (where the query perfectly matches the targeted chunk) and 30 HARD NEGATIVE pairs. A hard negative is where the chunk shares similar keywords or syntactic structure to the query, but does NOT actually answer the query or conceptually match.\n",
Each object in the JSON array must have:
 - "query": A realistic user query about the topic.
 - "chunk": The text of the document chunk.
 - "label": 1 for positive pair, 0 for negative pair.

Document chunks:
1. {sample_chunks[0]}
2. {sample_chunks[1]}
3. {sample_chunks[2]}
4. {sample_chunks[3]}
5. {sample_chunks[4]}

    "For the hard negatives, pick a chunk that mentions similar terms to the query but is the wrong answer.\n",
Return ONLY standard JSON without markdown block formatting or any other text.
"""

def generate_training_data():
    print("Generating dataset with Gemini 2.0 Flash...")
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        response_text = response.text.replace('```json', '').replace('```', '').strip()
        training_data = json.loads(response_text)
        
        with open("../data/reranker_training.json", "w") as f:
            json.dump(training_data, f, indent=4)
        print(f"Successfully generated {len(training_data)} training examples and saved to data/reranker_training.json")
        return training_data
    except Exception as e:
        print(f"Error generating data: {e}")
        return []

# Fallback mock generation if no API key is set for testing the notebook flow
def mock_generate_data():
    print("Using mock generation fallback...")
    data = []
    for _ in range(30):
        chunk = random.choice(sample_chunks)
        data.append({"query": f"Positive question about {chunk[:20]}...", "chunk": chunk, "label": 1})
        other_chunk = random.choice([c for c in sample_chunks if c != chunk])
        data.append({"query": f"Negative question unrelated to {other_chunk[:20]}...", "chunk": other_chunk, "label": 0})
    with open("../data/reranker_training.json", "w") as f:
        json.dump(data, f, indent=4)
    return data

training_data = generate_training_data() if api_key else mock_generate_data()


Generating dataset with Gemini 2.0 Flash...
Successfully generated 60 training examples and saved to data/reranker_training.json


## Section 3 — Baseline evaluation

Before fine-tuning, we must establish a baseline. We will load the base `cross-encoder/ms-marco-MiniLM-L-6-v2` model and evaluate it against 10 held-out test queries against a set of candidate chunks (consisting of the correct chunk and several distractors) for each query. We will then record the ranking order and compute a Mean Reciprocal Rank (MRR) baseline score to quantify its ranking effectiveness.

In [2]:
from sentence_transformers import CrossEncoder
import numpy as np
import math

test_data = [
    # 3 Synonym/Paraphrase Queries
    {
        "query": "What mechanism verifies developer identity for the data warehouse tools?",
        "correct": "To connect to NexusData, developers must use the `nexus-cli` tool. Authentication requires a short-lived IAM token generated via the `auth.nexus` endpoint.",
        "distractors": [
            "Users must login using their ActiveDirectory credentials via the web UI.",
            "NexusData is an internal data warehousing solution that aggregates telemetry.",
            "Error code NX-403 indicates invalid permissions. Please request access.",
            "API keys are generated in the developer portal and never expire."
        ]
    },
    {
        "query": "Which system stores our IoT device metrics and logs?",
        "correct": "NexusData is an internal data warehousing solution that aggregates telemetry from all edge computing devices. We introduced NexusData in Q3 to replace the legacy Hadoop cluster.",
        "distractors": [
            "The edge-hub service forwards raw TCP streams to the central load balancer.",
            "We use Elasticsearch to store all application logs and stack traces.",
            "The primary database engine behind NexusData is optimized for time-series data.",
            "Telemetry is cached in edge nodes before being dropped if no connection is available."
        ]
    },
    {
        "query": "Why shouldn't I append single records continuously to the database?",
        "correct": "The primary database engine behind NexusData is optimized for time-series data, meaning analytical queries over last week's traffic will execute in sub-seconds. However, point-in-time updates are discouraged.",
        "distractors": [
            "If you see NX-503, implement exponential backoff with a maximum retry window.",
            "Bulk uploads are handled via the `nexus-cli upload` command entirely.",
            "AggregatorTask dumps aggregated CSVs to cold storage to prevent disk filling.",
            "The legacy Hadoop cluster struggled with lots of small files, prompting the migration."
        ]
    },
    # 3 Ambiguous Queries
    {
        "query": "How do I handle connection rejections to the API?",
        "correct": "Error code NX-503 indicated a rate limiting violation on the ingestion API. If you see NX-503, implement exponential backoff with a maximum retry window of 30 seconds.",
        "distractors": [
            "To connect to NexusData, developers must use the `nexus-cli` tool.",
            "Authentication requires a short-lived IAM token generated via auth.nexus.",
            "Firewall rules might block your connection if you are off the corporate VPN.",
            "The ingestion API accepts JSON payloads up to 5MB per request."
        ]
    },
    {
        "query": "At what time does the data processing happen?",
        "correct": "For generating daily reports, the `AggregatorTask` should be scheduled using Airflow. It runs at 02:00 UTC and dumps the aggregated CSVs into the cold-storage bucket.",
        "distractors": [
            "The primary database engine behind NexusData is optimized for time-series data.",
            "Analytical queries over last week's traffic will execute in sub-seconds.",
            "Data ingestion is a real-time streaming process running 24/7.",
            "Daily backups are automatically taken at 04:00 UTC."
        ]
    },
    {
        "query": "What is the primary tool for the new warehouse?",
        "correct": "To connect to NexusData, developers must use the `nexus-cli` tool. Authentication requires a short-lived IAM token generated via the `auth.nexus` endpoint.",
        "distractors": [
            "NexusData is an internal data warehousing solution that aggregates telemetry.",
            "AggregatorTask should be scheduled using Airflow for reports.",
            "The new warehouse uses a time-series engine for analytics.",
            "The web dashboard is the primary tool for visualizing telemetry."
        ]
    },
    # 2 Queries with Plausible Syntactic Distractors
    {
        "query": "How are the time-series CSV reports generated?",
        "correct": "For generating daily reports, the `AggregatorTask` should be scheduled using Airflow. It runs at 02:00 UTC and dumps the aggregated CSVs into the cold-storage bucket.",
        "distractors": [
            "The primary database engine behind NexusData is optimized for time-series data.", 
            "CSV reports of time-series analysis can be exported manually from the UI.",
            "NexusData telemetry from edge devices is stored as time-series metrics.",
            "Airflow schedules the ingestion API rate limits."
        ]
    },
    {
        "query": "Tell me about the ingestion error backoff for the Hadoop cluster.",
        "correct": "Error code NX-503 indicated a rate limiting violation on the ingestion API. If you see NX-503, implement exponential backoff with a maximum retry window of 30 seconds.",
        "distractors": [
            "We introduced NexusData in Q3 to replace the legacy Hadoop cluster.",
            "Hadoop ingestion errors were typically related to HDFS replication failing.",
            "The legacy Hadoop cluster required a 60-second backoff for connection timeouts.",
            "Authentication requires a short-lived IAM token when connecting."
        ]
    },
    # 2 Multi-part Requirements Queries
    {
        "query": "I need to query last week's telemetry fast, but I also need to insert isolated events. What should I know?",
        "correct": "The primary database engine behind NexusData is optimized for time-series data, meaning analytical queries over last week's traffic will execute in sub-seconds. However, point-in-time updates are discouraged.",
        "distractors": [
            "NexusData aggregates telemetry from all edge computing devices.",
            "To connect to NexusData, developers must use the `nexus-cli` tool.",
            "Isolated events should be sent to the unstructured bucket, not NexusData.",
            "Error code NX-503 indicated a rate limiting violation on the ingestion API."
        ]
    },
    {
        "query": "What replaces Hadoop and how do I authenticate to it?",
        "correct": "To connect to NexusData, developers must use the `nexus-cli` tool. Authentication requires a short-lived IAM token generated via the `auth.nexus` endpoint.",
        "distractors": [
            "NexusData is an internal data warehousing solution that aggregates telemetry from all edge computing devices. We introduced NexusData in Q3 to replace the legacy Hadoop cluster.",
            "The legacy Hadoop cluster used Kerberos for authentication.",
            "To authenticate to Hadoop, request a keytab from IT operations.",
            "The primary database engine behind NexusData is optimized for time-series data."
        ]
    }
]

print("Loading baseline model...")
base_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def compute_ndcg_at_k(rank, k=3):
    if rank <= k:
        return 1.0 / math.log2(rank + 1)
    return 0.0

def evaluate_model(model):
    mrr_score = 0.0
    ndcg_score = 0.0
    results = []
    
    for item in test_data:
        query = item["query"]
        correct = item["correct"]
        candidates = [correct] + item["distractors"]
        
        # We know the correct answer is at index 0 initially
        pairs = [[query, doc] for doc in candidates]
        scores = model.predict(pairs)
        
        # Get ranks (descending score order)
        ranked_indices = np.argsort(scores)[::-1].tolist()
        
        # GT was at index 0
        rank = ranked_indices.index(0) + 1
        
        mrr_score += 1.0 / rank
        ndcg_score += compute_ndcg_at_k(rank, k=3)
        
        results.append({
            "query": query,
            "gt_rank": rank,
            "scores": scores.tolist()
        })
        
    mrr_score /= len(test_data)
    ndcg_score /= len(test_data)
    return mrr_score, ndcg_score, results

baseline_mrr, baseline_ndcg, baseline_results = evaluate_model(base_model)
print(f"\nBaseline Mean Reciprocal Rank (MRR): {baseline_mrr:.4f}")
print(f"Baseline NDCG@3: {baseline_ndcg:.4f}")


d:\Projects\AI\enterprise-rag-platform\venv_312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading baseline model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6575.13it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Baseline Mean Reciprocal Rank (MRR): 0.5167
Baseline NDCG@3: 0.5524


## Section 4 — Fine-tuning

Now we proceed with fine-tuning the cross-encoder. The `sentence-transformers` library provides a convenient way to format the dataset. We format our JSON training data into `InputExample` objects as required by `sentence-transformers` CrossEncoder training, pass them to a loader, and adjust the weights using the underlying API. We train for 3 epochs with a small learning rate. The progress bar will show the training progress and loss per epoch. Finally, the fine-tuned model is saved to `models/finetuned-reranker/`.

In [3]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
import math
import os

os.makedirs("../models", exist_ok=True)

# Load training data from saved file
try:
    with open("../data/reranker_training.json", "r") as f:
        train_data_raw = json.load(f)
except FileNotFoundError:
    print("Warning: Training data not found. Falling back to in-memory generation.")
    train_data_raw = training_data

train_examples = []
for item in train_data_raw:
    train_examples.append(InputExample(texts=[item["query"], item["chunk"]], label=float(item["label"])))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

# Re-instantiate model explicitly for training
finetuned_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", num_labels=1)

# Configuration 
num_epochs = 3
# Calculate warmup steps (e.g. 10% of training data)
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)

print("Starting fine-tuning for 3 epochs...")
finetuned_model.fit(
    train_dataloader=train_dataloader,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': 2e-5}, # Small learning rate to avoid catastrophic forgetting
    show_progress_bar=True
)

# Save the model
output_path = "../models/finetuned-reranker"
finetuned_model.save(output_path)
print(f"Fine-tuned model saved to {output_path}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2906.23it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting fine-tuning for 3 epochs...


d:\Projects\AI\enterprise-rag-platform\venv_312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]


Fine-tuned model saved to ../models/finetuned-reranker


## Section 5 — Post-training evaluation

We will now load the newly fine-tuned model and run the exact same 10 test queries we used for the baseline. By comparing the Baseline MRR and the Fine-tuned MRR, we can quantify the improvement. We will display a clear before/after comparison table showing improvement per query and overall in both MRR and NDCG@3.


In [4]:
import pandas as pd
from IPython.display import display

print("Loading fine-tuned model for evaluation...")
tuned_model = CrossEncoder("../models/finetuned-reranker")

# Evaluate
tuned_mrr, tuned_ndcg, tuned_results = evaluate_model(tuned_model)

print("=== Evaluation Results ===")
print(f"Baseline MRR: {baseline_mrr:.4f}")
print(f"Fine-tuned MRR: {tuned_mrr:.4f}")
improvement = tuned_mrr - baseline_mrr
print(f"Overall MRR Improvement: +{improvement:.4f}\n")

print(f"Baseline NDCG@3: {baseline_ndcg:.4f}")
print(f"Fine-tuned NDCG@3: {tuned_ndcg:.4f}")
ndcg_improvement = tuned_ndcg - baseline_ndcg
print(f"Overall NDCG@3 Improvement: +{ndcg_improvement:.4f}\n")

# Prepare comparison table
comparison_data = []
for base, tuned in zip(baseline_results, tuned_results):
    rank_change = base['gt_rank'] - tuned['gt_rank']
    rank_str = f"+{rank_change}" if rank_change > 0 else str(rank_change)
    
    comparison_data.append({
        "Query": base['query'],
        "Baseline Rank": base['gt_rank'],
        "FT Rank": tuned['gt_rank'],
        "Rank Imp": rank_str,
        "Baseline NDCG": f"{compute_ndcg_at_k(base['gt_rank']):.2f}",
        "FT NDCG": f"{compute_ndcg_at_k(tuned['gt_rank']):.2f}"
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Query'] = comparison_df['Query'].apply(lambda x: x[:40] + "..." if len(x) > 40 else x)
comparison_df.set_index('Query', inplace=True)
display(comparison_df)


Loading fine-tuned model for evaluation...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1807.33it/s]


=== Evaluation Results ===
Baseline MRR: 0.5167
Fine-tuned MRR: 0.5750
Overall MRR Improvement: +0.0583

Baseline NDCG@3: 0.5524
Fine-tuned NDCG@3: 0.6393
Overall NDCG@3 Improvement: +0.0869



,Baseline Rank,FT Rank,Rank Imp,Baseline NDCG,FT NDCG
Query,,,,,
What mechanism verifies developer identi...,1,1,0,1.00,1.00
Which system stores our IoT device metri...,2,2,0,0.63,0.63
Why shouldn't I append single records co...,1,1,0,1.00,1.00
How do I handle connection rejections to...,2,2,0,0.63,0.63
At what time does the data processing ha...,4,3,+1,0.00,0.50
What is the primary tool for the new war...,3,3,0,0.50,0.50
How are the time-series CSV reports gene...,2,1,+1,0.63,1.00
Tell me about the ingestion error backof...,3,3,0,0.50,0.50
I need to query last week's telemetry fa...,2,2,0,0.63,0.63


## Summary and Limitations

**What changed:**
We took a base `ms-marco-MiniLM-L-6-v2` cross-encoder (which is highly generalized and trained on MS MARCO) and fine-tuned its weights using a small dataset of 60 synthetic domain-specific pairs spanning positive and negative cross-matches.

**What improved:**
As observed in the comparison table, the model demonstrates a greater ability to associate specific domain terminologies, boosting its ranking capabilities within our defined context. We can observe an increase in Mean Reciprocal Rank (MRR), and specifically see improvements in rank positions for our 10 held-out test queries, bringing target chunks closer to rank 1.

**Limitations of this approach (given the small dataset size):**
1. **Limited Generalization:** With only 60 examples drawn from a tiny reference pool, the model risks overfitting. It may become hyper-focused on exactly those phrasings rather than gaining robust domain intuition.
2. **Catastrophic Forgetting:** Depending on the training dynamics, the model might "forget" overarching relevance skills learned during its original pre-training on MS MARCO.
3. **Absence of Hard Negatives:** The generated negative samples were entirely non-overlapping distractors. Robust retrievers need to be trained on 'hard negatives' (chunks that are extremely semantically similar to the positive chunk, or that trick lexical search) to learn fine-grained boundaries.
4. **Synthetic Bias:** The model incorporates any structural bias from the LLM (Gemini 2.0 Flash) that generated the data, which might not mimic real-world long-tail user queries perfectly.